# ABFM Skills Assessment - R Notebook
## Quick-Deploy for Synthetic EHR Analysis

# ================================================================
# SECTION 0 — PACKAGES
# ================================================================

In [3]:
# ╔════════════════════════════════════════════════════════════════╗
# ║  0A. INSTALL ANY MISSING PACKAGES (run once)                   ║
# ╚════════════════════════════════════════════════════════════════╝

required <- c(
  # ── Core data wrangling ──
  "tidyverse",   # dplyr, ggplot2, tidyr, readr, stringr, forcats, purrr, lubridate
  "data.table",  # fast CSV reads + large-data joins

  # ── EDA & data quality ──
  "naniar",      # missing data visualisation (gg_miss_var, vis_miss)
  "visdat",      # vis_dat() whole-dataset type overview
  "tableone",    # Table 1 demographics summary with p-values
  "DescTools",   # misc descriptive utilities
  "corrplot",    # correlation matrix heatmap
  "ggcorrplot",  # ggplot2-style correlation heatmap
  "UpSetR",      # upset plots for set intersections (comorbidity overlap)
  "cowplot",     # multi-panel ggplot layouts
  "gridExtra",   # grid.arrange for side-by-side plots
  "ggpubr",      # publication-ready ggplot themes + stat_compare_means
  "ggrepel",     # non-overlapping text labels in ggplot
  "scales",      # percent(), comma() for axis formatting
  "viridis",     # colourblind-friendly palettes
  "car",         # vif() for multicollinearity check

  # ── General linear / generalised linear models ──
  "MASS",        # glm.nb() negative binomial; stepAIC
  "glmnet",      # LASSO / elastic net regularised GLM
  "mgcv",        # gam() generalised additive models
  "statmod",     # tweedie() distribution family for GLM cost models
  "pscl",        # zeroinfl() zero-inflated Poisson / NB
  "sandwich",    # HC robust standard errors

  # ── Mixed-effects / multilevel models ──
  "lme4",        # lmer(), glmer() — the workhorse for nested/clustered data
  "lmerTest",    # adds p-values to lmer via Satterthwaite
  "performance", # icc(), model_performance(), check_model()
  "broom.mixed", # tidy() for lme4 objects
  "broom",       # tidy(), glance(), augment() for all model families

  # ── Bayesian methods ──
  "brms",        # Bayesian mixed models (Stan backend)
  "rstanarm",    # stan_glmer() — simpler Bayesian mixed models
  "bayestestR",  # describe_posterior(), rope(), equivalence tests
  "bayesplot",   # mcmc_trace, mcmc_areas for MCMC diagnostics

  # ── Survival / time-to-event ──
  "survival",    # Surv(), survfit(), coxph(), survreg()
  "survminer",   # ggsurvplot() — KM curves with risk tables
  "flexsurv",    # flexible parametric survival (Weibull, Gompertz, spline)

  # ── Machine learning: tree methods ──
  "randomForest", # classic RF
  "ranger",       # fast RF (preferred for large n)
  "xgboost",      # gradient boosted trees
  "lightgbm",     # Microsoft's gradient boosting (fast + handles categoricals)
  "gbm",          # Friedman's GBM
  "rpart",        # single decision tree
  "rpart.plot",   # pretty tree visualisation
  "C50",          # C5.0 decision tree + boosted ensemble

  # ── Machine learning: framework / tuning ──
  "caret",        # unified ML training framework (train(), confusionMatrix())
  "yardstick",    # metric functions (roc_auc, rmse, etc.)
  "pROC",         # ROC curves + AUC with CI
  "Metrics",      # rmse(), mae(), auc(), logloss()
  "vip",          # variable importance plots (model-agnostic)
  "recipes",      # tidymodels preprocessing pipelines

  # ── Propensity scores / causal inference ──
  "MatchIt",      # propensity score matching (nearest, optimal, CEM)

  # ── Time series ──
  "zoo",          # zoo time series objects, rollmean()
  "xts",          # extensible time series

  # ── Misc ──
  "e1071",        # SVM + skewness/kurtosis functions
  "rstatix"       # pipe-friendly stats (t_test, anova_test, etc.)
)

# Only install what's missing
missing <- required[!(required %in% installed.packages()[,"Package"])]
if (length(missing) > 0) {
  cat("Installing:", paste(missing, collapse=", "), "\n")
  install.packages(missing, repos="https://cloud.r-project.org", quiet=TRUE)
} else {
  cat("All", length(required), "packages already installed. Good to go.\n")
}

All 53 packages already installed. Good to go.


In [ ]:
# ╔════════════════════════════════════════════════════════════════╗
# ║  0B. LOAD LIBRARIES                                          ║
# ╚════════════════════════════════════════════════════════════════╝

suppressPackageStartupMessages({

  # ── Core data wrangling ──
  library(tidyverse)    # dplyr ggplot2 tidyr readr stringr forcats purrr lubridate
  library(data.table)

  # ── EDA & data quality ──
  library(naniar)       # gg_miss_var, vis_miss
  library(visdat)       # vis_dat
  library(tableone)     # CreateTableOne
  library(DescTools)    # Desc, MeanCI, etc.
  library(corrplot)     # corrplot()
  library(ggcorrplot)   # ggcorrplot()
  library(UpSetR)       # upset()
  library(cowplot)      # plot_grid()
  library(gridExtra)    # grid.arrange()
  library(ggpubr)       # ggarrange, stat_compare_means
  library(ggrepel)      # geom_text_repel
  library(scales)       # percent, comma
  library(viridis)      # scale_fill_viridis
  library(car)          # vif()

  # ── GLM / regression ──
  library(MASS)         # glm.nb, stepAIC
  library(glmnet)       # cv.glmnet
  library(mgcv)         # gam, s()
  library(statmod)      # tweedie family
  library(pscl)         # zeroinfl
  library(sandwich)     # vcovHC

  # ── Mixed-effects ──
  library(lme4)         # glmer
  library(lmerTest)     # p-values for lmer
  library(performance)  # icc(), r2()
  library(broom)        # tidy
  library(broom.mixed)  # tidy for lme4

  # ── Bayesian ──
  # library(brms)       # uncomment if needed — slow to load
  # library(rstanarm)   # uncomment if needed — slow to load
  library(bayestestR)   # describe_posterior

  # ── Survival ──
  library(survival)     # Surv, survfit, coxph, survreg
  library(survminer)    # ggsurvplot
  library(flexsurv)     # flexsurvreg

  # ── ML tree methods ──
  library(randomForest)
  library(ranger)
  library(xgboost)
  library(gbm)
  library(rpart)
  library(rpart.plot)

  # ── ML framework / metrics ──
  library(caret)        # train, confusionMatrix, createDataPartition
  library(pROC)         # roc, auc, ci.auc
  library(Metrics)      # rmse, mae
  library(vip)          # vip() variable importance

  # ── Propensity matching ──
  library(MatchIt)

  # ── Time series ──
  library(zoo)

  # ── Misc ──
  library(e1071)        # skewness, kurtosis, SVM
  library(rstatix)      # pipe-friendly stats
})

cat("All libraries loaded.\n")

# ================================================================
# SECTION 1 — DATA LOADING
# ================================================================

In [ ]:
# ╔════════════════════════════════════════════════════════════════╗
# ║  1A. LOAD ALL 8 TABLES                                         ║
# ╚════════════════════════════════════════════════════════════════╝

DATA_PATH <- "data/"

# Auto-detect all CSV files in the folder
csv_files <- list.files(DATA_PATH, pattern = "\\.csv$", full.names = TRUE)
cat("Found", length(csv_files), "CSV files:\n")
cat(paste(" ", basename(csv_files)), sep = "\n")

# Read each into a named list AND assign to individual variables
all_tables <- list()
for (f in csv_files) {
  nm <- tools::file_path_sans_ext(basename(f))
  df <- read_csv(f, show_col_types = FALSE)
  all_tables[[nm]] <- df
  all_tables_original[[nm]] <- df
  assign(nm, df, envir = .GlobalEnv)
  assign(paste0(nm,"_original"), df, envir = .GlobalEnv)
  cat(sprintf("  %-25s: %6s rows  x  %2d cols\n",
              nm, format(nrow(df), big.mark=","), ncol(df)))
}

cat("\nAll tables loaded into global env by their file name.\n")
cat("\nOriginal tables loaded by their file name _original.\n")
cat("Also accessible via all_tables[['table_name']].\n")

# ================================================================
# SECTION 2 — EXPLORATORY DATA ANALYSIS
# ================================================================

# ================================================================
# SECTION 3 — FEATURE ENGINEERING
# ================================================================

# ================================================================
# SECTION 4 — ANSWERS
# ================================================================

In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════╗
# ║  ANSWER 1                                                                  ║
# ╚════════════════════════════════════════════════════════════════════════════╝

# QUESTION: (paste question here)
#

# CODE:



INTERPRETATION:


In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════╗
# ║  ANSWER 2                                                                  ║
# ╚════════════════════════════════════════════════════════════════════════════╝

# QUESTION: (paste question here)
#

# CODE:



INTERPRETATION:


In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════╗
# ║  ANSWER 3                                                                  ║
# ╚════════════════════════════════════════════════════════════════════════════╝

# QUESTION: (paste question here)
#

# CODE:



INTERPRETATION:


In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════╗
# ║  ANSWER 4                                                                  ║
# ╚════════════════════════════════════════════════════════════════════════════╝

# QUESTION: (paste question here)
#

# CODE:



INTERPRETATION:


In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════╗
# ║  ANSWER 5                                                                  ║
# ╚════════════════════════════════════════════════════════════════════════════╝

# QUESTION: (paste question here)
#

# CODE:



INTERPRETATION:


In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════╗
# ║  ANSWER 6                                                                  ║
# ╚════════════════════════════════════════════════════════════════════════════╝

# QUESTION: (paste question here)
#

# CODE:



INTERPRETATION:


In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════╗
# ║  ANSWER 7                                                                  ║
# ╚════════════════════════════════════════════════════════════════════════════╝

# QUESTION: (paste question here)
#

# CODE:



INTERPRETATION:


In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════╗
# ║  ANSWER 8                                                                  ║
# ╚════════════════════════════════════════════════════════════════════════════╝

# QUESTION: (paste question here)
#

# CODE:



INTERPRETATION:


In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════╗
# ║  ANSWER 9                                                                  ║
# ╚════════════════════════════════════════════════════════════════════════════╝

# QUESTION: (paste question here)
#

# CODE:



INTERPRETATION:


In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════╗
# ║  ANSWER 10                                                                 ║
# ╚════════════════════════════════════════════════════════════════════════════╝

# QUESTION: (paste question here)
#

# CODE:



INTERPRETATION:
